# Exercises & Coding Challenges

This notebook contains standalone coding challenges that integrate concepts from multiple modules. Each challenge deepens your understanding through implementation.

## Challenge 1: Build a Complete Mini-Swarm from Scratch

**Difficulty: Medium | Time: 60-90 minutes**

Implement a complete, simplified Model Swarms system that works on 2D weight vectors. This lets you study the algorithm's dynamics without needing GPUs or large models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Callable

def multi_peak_utility(x: np.ndarray) -> float:
    """
    A synthetic utility function with multiple peaks.
    Simulates a weight-space landscape with several 'good' regions.
    Global maximum at approximately (2.0, 1.5).
    """
    return (
        0.8 * np.exp(-((x[0]-2)**2 + (x[1]-1.5)**2) / 0.5)    # global peak
        + 0.6 * np.exp(-((x[0]+1)**2 + (x[1]-0.5)**2) / 0.8)  # local peak 1
        + 0.5 * np.exp(-((x[0]-0.5)**2 + (x[1]+1)**2) / 0.6)  # local peak 2
        + 0.3 * np.exp(-((x[0]+1.5)**2 + (x[1]+1.5)**2) / 1.0) # local peak 3
    )

# Visualize the landscape
x = np.linspace(-3, 4, 200)
y = np.linspace(-3, 4, 200)
X, Y = np.meshgrid(x, y)
Z = np.array([[multi_peak_utility(np.array([xi, yi])) for xi, yi in zip(xrow, yrow)]
               for xrow, yrow in zip(X, Y)])

fig, ax = plt.subplots(figsize=(8, 6))
c = ax.contourf(X, Y, Z, levels=30, cmap='viridis')
plt.colorbar(c, label='Utility')
ax.scatter(2.0, 1.5, c='red', s=200, marker='*', zorder=5, label='Global optimum')
ax.legend()
ax.set_title('Multi-Peak Utility Landscape')
ax.set_xlabel('Weight dimension 1')
ax.set_ylabel('Weight dimension 2')
plt.tight_layout()
plt.show()

In [ ]:
class MiniModelSwarm:
    """
    A simplified Model Swarms implementation operating on 2D weight vectors.
    """

    def __init__(
        self,
        utility_func: Callable[[np.ndarray], float],
        initial_positions: List[np.ndarray],
        target_particles: int = 15,
        inertia: float = 0.2,
        cognitive_coeff: float = 0.3,
        social_coeff: float = 0.4,
        repel_coeff: float = 0.1,
        step_length: float = 1.0,
        step_decay: float = 0.95,
        min_step: float = 0.1,
        patience: int = 10,
        restart_patience_ratio: float = 0.67,
        use_randomness: bool = True,
    ):
        self.utility_func = utility_func
        self.inertia = inertia
        self.cognitive_coeff = cognitive_coeff
        self.social_coeff = social_coeff
        self.repel_coeff = repel_coeff
        self.step_length = step_length
        self.step_decay = step_decay
        self.min_step = min_step
        self.patience = patience
        self.restart_patience = int(restart_patience_ratio * patience)
        self.use_randomness = use_randomness

        # TODO: Population expansion
        # Expand initial_positions to target_particles via pairwise interpolation
        # t ~ Uniform(0, 2), child = t*p1 + (1-t)*p2
        self.positions = list(initial_positions)
        while len(self.positions) < target_particles:
            idx1, idx2 = np.random.choice(len(initial_positions), 2, replace=(len(initial_positions) < 2))
            t = np.random.random() * 2
            child = t * initial_positions[idx1] + (1-t) * initial_positions[idx2]
            self.positions.append(child)
        self.positions = np.array(self.positions)
        self.n_particles = len(self.positions)

        # TODO: Initialize velocities (random mode)
        self.velocities = np.zeros_like(self.positions)
        for i in range(self.n_particles):
            j = np.random.randint(self.n_particles)
            while j == i:
                j = np.random.randint(self.n_particles)
            self.velocities[i] = self.positions[j] - self.positions[i]

        # TODO: Evaluate all particles
        self.scores = np.array([utility_func(p) for p in self.positions])

        # TODO: Set up tracking
        self.personal_best_positions = self.positions.copy()
        self.personal_best_scores = self.scores.copy()
        self.stagnation_counters = np.zeros(self.n_particles, dtype=int)

        best_idx = np.argmax(self.scores)
        self.global_best_position = self.positions[best_idx].copy()
        self.global_best_score = self.scores[best_idx]

        worst_idx = np.argmin(self.scores)
        self.global_worst_position = self.positions[worst_idx].copy()
        self.global_worst_score = self.scores[worst_idx]

        self.history = {
            'global_best_scores': [self.global_best_score],
            'positions': [self.positions.copy()],
            'step_lengths': [self.step_length],
        }

    def step(self) -> dict:
        """Execute one iteration of Model Swarms."""
        improved = False
        restarts = 0

        for i in range(self.n_particles):
            # Check restart
            if self.stagnation_counters[i] >= self.restart_patience:
                self.positions[i] = self.personal_best_positions[i].copy()
                self.velocities[i] = np.zeros_like(self.velocities[i])
                self.stagnation_counters[i] = 0
                restarts += 1
                continue

            # Sample random coefficients
            if self.use_randomness:
                r_v, r_p, r_g, r_w = np.random.random(4)
            else:
                r_v = r_p = r_g = r_w = 1.0

            # Normalize weights
            w_i = r_v * self.inertia
            w_c = r_p * self.cognitive_coeff
            w_s = r_g * self.social_coeff
            w_r = r_w * self.repel_coeff
            C = w_i + w_c + w_s + w_r
            w_i, w_c, w_s, w_r = w_i/C, w_c/C, w_s/C, w_r/C

            # Velocity update
            self.velocities[i] = (
                w_i * self.velocities[i]
                + w_c * (self.personal_best_positions[i] - self.positions[i])
                + w_s * (self.global_best_position - self.positions[i])
                + w_r * (self.positions[i] - self.global_worst_position)
            )

            # Position update
            self.positions[i] += self.step_length * self.velocities[i]

            # Evaluate
            score = self.utility_func(self.positions[i])
            self.scores[i] = score

            # Update personal best
            if score > self.personal_best_scores[i]:
                self.personal_best_scores[i] = score
                self.personal_best_positions[i] = self.positions[i].copy()
                self.stagnation_counters[i] = 0

                if score > self.global_best_score:
                    self.global_best_score = score
                    self.global_best_position = self.positions[i].copy()
                    improved = True
            else:
                self.stagnation_counters[i] += 1

            if score < self.global_worst_score:
                self.global_worst_score = score
                self.global_worst_position = self.positions[i].copy()

        # Decay step length
        self.step_length = max(self.step_length * self.step_decay, self.min_step)

        self.history['global_best_scores'].append(self.global_best_score)
        self.history['positions'].append(self.positions.copy())
        self.history['step_lengths'].append(self.step_length)

        return {'global_best_score': self.global_best_score,
                'improved': improved, 'restarts': restarts}

    def search(self, max_iter: int = 50) -> Tuple[np.ndarray, float]:
        """Run full search with patience-based stopping."""
        no_improve_count = 0

        for iteration in range(max_iter):
            result = self.step()

            if result['improved']:
                no_improve_count = 0
            else:
                no_improve_count += 1

            if no_improve_count >= self.patience:
                print(f"Patience reached at iteration {iteration + 1}")
                break
        else:
            print(f"Max iterations ({max_iter}) reached")

        return self.global_best_position, self.global_best_score

In [ ]:
# Run the mini swarm!
np.random.seed(42)

initial_experts = [
    np.array([-1.0, 0.5]),   # near local peak 1
    np.array([0.5, -1.0]),   # near local peak 2
    np.array([-1.5, -1.5]),  # near local peak 3
    np.array([1.0, 0.0]),    # between peaks
    np.array([0.0, 1.0]),    # between peaks
]

swarm = MiniModelSwarm(
    utility_func=multi_peak_utility,
    initial_positions=initial_experts,
    target_particles=15,
)

best_pos, best_score = swarm.search(max_iter=30)
print(f"\nBest position: [{best_pos[0]:.3f}, {best_pos[1]:.3f}]")
print(f"Best score: {best_score:.4f}")
print(f"True optimum: [2.0, 1.5] with score ~0.8")
print(f"Distance to optimum: {np.linalg.norm(best_pos - np.array([2.0, 1.5])):.3f}")

In [ ]:
# Visualize the search trajectory
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: particle trajectories on landscape
x = np.linspace(-3, 4, 200)
y = np.linspace(-3, 4, 200)
X, Y = np.meshgrid(x, y)
Z = np.array([[multi_peak_utility(np.array([xi, yi])) for xi, yi in zip(xrow, yrow)]
               for xrow, yrow in zip(X, Y)])

ax1.contourf(X, Y, Z, levels=30, cmap='viridis', alpha=0.6)

# Plot initial positions
init_pos = swarm.history['positions'][0]
ax1.scatter(init_pos[:5, 0], init_pos[:5, 1], c='blue', s=80, marker='o',
           zorder=5, label='Initial experts', edgecolors='white')
ax1.scatter(init_pos[5:, 0], init_pos[5:, 1], c='cyan', s=40, marker='x',
           zorder=5, label='Interpolated')

# Plot final positions
final_pos = swarm.history['positions'][-1]
ax1.scatter(final_pos[:, 0], final_pos[:, 1], c='red', s=50, marker='^',
           zorder=5, label='Final positions', edgecolors='white')

# Mark global best
ax1.scatter(best_pos[0], best_pos[1], c='yellow', s=200, marker='*',
           zorder=6, label=f'Found best ({best_score:.3f})', edgecolors='black')
ax1.scatter(2.0, 1.5, c='white', s=200, marker='*',
           zorder=6, label='True optimum', edgecolors='black')

ax1.set_title('Swarm Search Trajectory')
ax1.legend(fontsize=8, loc='lower left')
ax1.set_xlim(-3, 4)
ax1.set_ylim(-3, 4)

# Right: convergence curve
ax2.plot(swarm.history['global_best_scores'], 'b-', linewidth=2, label='Global best')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Utility Score')
ax2.set_title('Convergence Curve')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Challenge 2: Merge Operation with Verification

**Difficulty: Easy-Medium | Time: 30 minutes**

We already implemented and tested `lora_merge` in Module 3. Here's a more thorough test suite:

In [ ]:
import os
import tempfile
from safetensors.numpy import load_file
from model_swarms_course.merging import lora_merge_from_paths as lora_merge, create_dummy_adapter

print("Imported LoRA merge helpers from src/model_swarms_course/merging.py")


## Challenge 3: Hyperparameter Sensitivity Visualization

**Difficulty: Medium | Time: 45 minutes**

Use the MiniModelSwarm to explore hyperparameter sensitivity:

In [ ]:
def run_sensitivity_experiment(param_name, param_values, n_trials=5, **base_kwargs):
    """Run mini swarm with different values of one parameter."""
    results = {}
    for val in param_values:
        scores = []
        for trial in range(n_trials):
            np.random.seed(trial * 100)
            kwargs = base_kwargs.copy()
            kwargs[param_name] = val
            swarm = MiniModelSwarm(
                utility_func=multi_peak_utility,
                initial_positions=[
                    np.array([-1.0, 0.5]), np.array([0.5, -1.0]),
                    np.array([-1.5, -1.5]), np.array([1.0, 0.0]),
                    np.array([0.0, 1.0]),
                ],
                **kwargs
            )
            _, score = swarm.search(max_iter=30)
            scores.append(score)
        results[val] = (np.mean(scores), np.std(scores))
    return results

# Run experiments
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

params = [
    ('inertia', [0.05, 0.1, 0.2, 0.4, 0.6]),
    ('cognitive_coeff', [0.1, 0.2, 0.3, 0.5, 0.7]),
    ('social_coeff', [0.1, 0.2, 0.4, 0.6, 0.8]),
    ('repel_coeff', [0.0, 0.05, 0.1, 0.2, 0.3]),
]

for ax, (param, values) in zip(axes.flat, params):
    results = run_sensitivity_experiment(param, values)
    means = [results[v][0] for v in values]
    stds = [results[v][1] for v in values]
    ax.errorbar(values, means, yerr=stds, marker='o', capsize=5, linewidth=2)
    ax.set_xlabel(param)
    ax.set_ylabel('Final Best Score')
    ax.set_title(f'Effect of {param}')
    ax.grid(True, alpha=0.3)

plt.suptitle('Hyperparameter Sensitivity Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Challenge 4: Step Length Schedule Comparison

**Difficulty: Medium | Time: 30 minutes**

In [ ]:
# Compare different step length schedules
schedules = {
    'Fixed (1.0)': 1.0,
    'Slow decay (0.98)': 0.98,
    'Default (0.95)': 0.95,
    'Fast decay (0.85)': 0.85,
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, decay in schedules.items():
    scores_all = []
    for trial in range(5):
        np.random.seed(trial * 100)
        swarm = MiniModelSwarm(
            utility_func=multi_peak_utility,
            initial_positions=[
                np.array([-1.0, 0.5]), np.array([0.5, -1.0]),
                np.array([-1.5, -1.5]), np.array([1.0, 0.0]),
                np.array([0.0, 1.0]),
            ],
            step_decay=decay,
            patience=15,
        )
        swarm.search(max_iter=40)
        scores_all.append(swarm.history['global_best_scores'])

    # Pad shorter runs
    max_len = max(len(s) for s in scores_all)
    for s in scores_all:
        while len(s) < max_len:
            s.append(s[-1])

    mean_curve = np.mean(scores_all, axis=0)
    ax1.plot(mean_curve, label=name, linewidth=2)

ax1.set_xlabel('Iteration')
ax1.set_ylabel('Global Best Score')
ax1.set_title('Convergence by Step Decay Rate')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Show step length schedules
iters = np.arange(40)
for name, decay in schedules.items():
    ax2.plot(iters, np.maximum(decay**iters, 0.1), label=name, linewidth=2)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Step Length')
ax2.set_title('Step Length Schedules')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Challenge 5: Diversity Experiment

**Difficulty: Medium | Time: 30 minutes**

Reproduce the paper's diversity finding with the mini swarm:

In [ ]:
# How does initial expert diversity affect search performance?
# Fix total particles at 15. Vary number of distinct starting experts.

configs = [
    ("1 expert \u00d7 15", 1),
    ("3 experts \u00d7 5", 3),
    ("5 experts \u00d7 3", 5),
    ("5 experts + 10 interp.", 5),  # this is our default
]

all_experts = [
    np.array([-1.0, 0.5]), np.array([0.5, -1.0]),
    np.array([-1.5, -1.5]), np.array([1.0, 0.0]),
    np.array([0.0, 1.0]),
]

results = {}
for name, n_distinct in configs:
    trial_scores = []
    for trial in range(10):
        np.random.seed(trial * 42)
        experts = [all_experts[i % n_distinct].copy() for i in range(n_distinct)]
        # Add small noise to duplicates
        for i in range(len(experts)):
            experts[i] = experts[i] + np.random.randn(2) * 0.01

        swarm = MiniModelSwarm(
            utility_func=multi_peak_utility,
            initial_positions=experts,
            target_particles=15,
        )
        _, score = swarm.search(max_iter=30)
        trial_scores.append(score)
    results[name] = (np.mean(trial_scores), np.std(trial_scores))

fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
means = [results[n][0] for n in names]
stds = [results[n][1] for n in names]
bars = ax.bar(names, means, yerr=stds, capsize=8, color=['#EF5350', '#FFA726', '#66BB6A', '#42A5F5'])
ax.set_ylabel('Final Best Score')
ax.set_title('Effect of Expert Diversity on Search Performance (15 total particles)')
ax.grid(True, alpha=0.3, axis='y')

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{mean:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## Challenge 6: Ablation Study

**Difficulty: Medium | Time: 30 minutes**

Systematically remove components and measure the impact:

In [ ]:
# Ablation study: remove one component at a time
ablations = {
    'Full Model Swarms': dict(),
    'No randomness': dict(use_randomness=False),
    'No repulsion': dict(repel_coeff=0.0),
    'No inertia': dict(inertia=0.0),
    'No step decay': dict(step_decay=1.0),
    'High inertia': dict(inertia=0.6, cognitive_coeff=0.1, social_coeff=0.2),
}

results = {}
for name, overrides in ablations.items():
    trial_scores = []
    for trial in range(10):
        np.random.seed(trial * 42)
        kwargs = dict(
            utility_func=multi_peak_utility,
            initial_positions=[
                np.array([-1.0, 0.5]), np.array([0.5, -1.0]),
                np.array([-1.5, -1.5]), np.array([1.0, 0.0]),
                np.array([0.0, 1.0]),
            ],
            target_particles=15,
        )
        kwargs.update(overrides)
        swarm = MiniModelSwarm(**kwargs)
        _, score = swarm.search(max_iter=30)
        trial_scores.append(score)
    results[name] = (np.mean(trial_scores), np.std(trial_scores))

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
names = list(results.keys())
means = [results[n][0] for n in names]
stds = [results[n][1] for n in names]
colors = ['#4CAF50'] + ['#FF9800'] * (len(names) - 1)
bars = ax.bar(names, means, yerr=stds, capsize=5, color=colors)
ax.set_ylabel('Final Best Score')
ax.set_title('Ablation Study: Impact of Removing Components')
ax.set_xticklabels(names, rotation=15, ha='right')
ax.grid(True, alpha=0.3, axis='y')

# Add baseline reference line
ax.axhline(y=means[0], color='green', linestyle='--', alpha=0.5, label='Full model baseline')
ax.legend()

plt.tight_layout()
plt.show()

print("\nAblation Results:")
baseline = means[0]
for name, (mean, std) in results.items():
    diff = mean - baseline
    print(f"  {name:<25s}: {mean:.4f} \u00b1 {std:.4f}  ({diff:+.4f})")

## Self-Assessment Rubric

| Criterion | Points |
|-----------|--------|
| **Correctness**: Code runs and produces correct results | 40% |
| **Completeness**: All features implemented | 20% |
| **Understanding**: Comments show you understand WHY, not just HOW | 25% |
| **Code quality**: Clean, well-structured, Pythonic | 15% |

### Minimum Passing (70%):
- Challenges 1-2 fully working
- Challenge 3 at least partially working

### Distinction (90%+):
- All 6 challenges working with visualizations
- Ablation study reveals non-obvious insights

---

**[Back to Course Overview](README.md)**